# Protein Binder Metric Evaluation Stats
Data loading, metric evaluation (ROC-AUC / PR-AUC / F1 / precision), effect sizes (Cliff's delta, AUROC), and redundancy (Spearman / PCA / UMAP). Bootstrap subsampling has been removed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import config

# Repoint this at your merged.csv (raw). No 'iteration' column is needed
# any more -- bootstrap subsampling has been removed from the analysis.
df = pd.read_csv("/home/bri/pymol/project_offline/project_current/data/raw/alphaseq/merged.csv")
YAML_PATH  = Path(config.YAML_PATH)
OUTPUT_DIR = Path(config.OUT_DIR)

df.head()

NameError: name 'config' is not defined

## stuff to define in constants or utils

In [ ]:
# TO DO: already defined on 03_rankings.ipynb, backtrace
# numeric metric columns: numeric dtype, minus meta/excluded columns.
# meta_cols is defined first so numeric_cols is derived from it,
# not the other way round. KD[M] / EC50[M] stay excluded.
import config
meta_cols    = config.META_COLS
exclude_cols = config.EXCLUDE_COLS

numeric_cols = [
    c for c in df.select_dtypes(include="number").columns
    if c not in meta_cols and c not in exclude_cols
]

print(f"{len(numeric_cols)} numeric metric columns selected")
numeric_cols


In [ ]:
# TO DO: do we need this? It is already defined in 03_rankings.ipynb, backtrace
# # Load metric directionality from YAML
# +1 → higher is better   -1 → lower is better
if Path(YAML_PATH).exists():
    with open(YAML_PATH, 'r') as f:
        yaml_data = yaml.safe_load(f)
    raw_directions = {k: v['direction'] for k, v in yaml_data['metrics'].items()}
    # Sort longest-key-first so 'iptm' is checked before 'ptm'
    metric_directions = dict(
        sorted(raw_directions.items(), key=lambda x: len(x[0]), reverse=True)
    )
    print(f"Loaded {len(metric_directions)} metric directions from YAML.")
else:
    print(f"WARNING: YAML not found at {YAML_PATH}. Using default direction (+1).")
    metric_directions = {}


def get_direction(col_name):
    """Return +1 or -1 for a metric column.

    Uses suffix matching (col ends with '_<base>') to avoid 'ptm'
    accidentally matching 'iptm'.
    """
    for base_metric, direction in metric_directions.items():
        if col_name == base_metric or col_name.endswith(f'_{base_metric}'):
            return direction
    return 1  # default: higher is better

#### Compute metrics (ROC-AUC, PR-AUC, F1, Precision)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve,
    average_precision_score, f1_score
)

def calculate_all_metrics(df_data, numeric_cols):
    """
    Calculate ROC-AUC, PR-AUC, and F1 at the PR-curve-optimal threshold
    for each predictor. Direction is resolved via get_direction() from YAML.

    Threshold logic:
        - precision_recall_curve is computed on aligned scores (higher = better)
        - F1 is maximized across all PR-curve thresholds
        - opt_threshold_aligned: threshold in aligned score space
        - opt_threshold_raw: threshold in original score space (for inference)
    """
    results = []

    for col in numeric_cols:
        if df_data[col].notna().sum() < 2 or df_data[col].nunique() < 2:
            continue

        try:
            clean_data = df_data[[col, 'binder']].dropna()
            if clean_data['binder'].nunique() < 2:
                continue

            y_true        = clean_data['binder'].astype(int)
            y_score_raw   = clean_data[col].values
            direction     = get_direction(col)
            aligned_scores = y_score_raw * direction

            # ROC-AUC
            raw_roc     = roc_auc_score(y_true, y_score_raw)
            aligned_roc = max(raw_roc, 1 - raw_roc)

            # PR-AUC
            pr_auc = average_precision_score(y_true, aligned_scores)

            # F1-optimal threshold from PR curve
            precisions, recalls, thresholds = precision_recall_curve(y_true, aligned_scores)
            # precisions/recalls have one extra point (the (0,1) endpoint) → use [:-1]
            f1_scores = (
                2 * precisions[:-1] * recalls[:-1]
            ) / (precisions[:-1] + recalls[:-1] + 1e-8)

            # Guard: degenerate metric
            if f1_scores.max() == 0:
                results.append({
                    'metric': col, 'raw_roc': round(raw_roc, 4),
                    'aligned_roc': round(aligned_roc, 4), 'pr_auc': round(pr_auc, 4),
                    'f1': 0.0, 'precision': 0.0, 'recall': 0.0,
                    'opt_threshold': np.nan, 'opt_threshold_raw': np.nan,
                    'direction': direction, 'note': 'degenerate_f1'
                })
                continue

            best_idx  = np.argmax(f1_scores)
            best_thr  = thresholds[best_idx]           # in aligned space
            y_pred    = (aligned_scores >= best_thr).astype(int)

            results.append({
                'metric':               col,
                'raw_roc':              round(raw_roc, 4),
                'aligned_roc':          round(aligned_roc, 4),
                'pr_auc':               round(pr_auc, 4),
                'f1':                   round(f1_score(y_true, y_pred), 4),
                'precision':            round(float(precisions[:-1][best_idx]), 4),
                'recall':               round(float(recalls[:-1][best_idx]), 4),
                'opt_threshold': round(float(best_thr), 4),          # aligned space
                'opt_threshold_raw':     round(float(best_thr * direction), 4), # original space
                'direction':            direction,
                'note':                 '',
            })

        except Exception as e:
            results.append({'metric': col, 'note': f'error: {e}'})
            continue

    return pd.DataFrame(results).sort_values('pr_auc', ascending=False)

In [ ]:
# Single evaluation over the full table (no bootstrap / iterations).
rankings = calculate_all_metrics(df, numeric_cols)
rankings.to_csv(OUTPUT_DIR / "rankings.csv", index=False)

print(f"{len(rankings)} metrics evaluated")
rankings.head(15)

### Visualization

In [ ]:
# Top 15 metrics by PR-AUC (no error bars -- single run)
top_15 = rankings.head(15).copy()

fig, ax = plt.subplots(figsize=(16, 8))
x_pos = np.arange(len(top_15))
width = 0.2

metrics_to_plot = [
    ("aligned_roc", "ROC-AUC", "#e74c3c"),
    ("pr_auc", "PR-AUC", "#2ecc71"),
    ("f1", "F1-Score", "#3498db"),
    ("precision", "Precision", "#f39c12"),
]

for i, (col, label, color) in enumerate(metrics_to_plot):
    ax.bar(x_pos + (i - 1.5) * width, top_15[col], width, label=label, color=color, alpha=0.85)

ax.set_ylabel("Score")
ax.set_title("Top 15 Metrics by PR-AUC", fontweight="bold")
ax.set_xticks(x_pos)
ax.set_xticklabels(top_15["metric"], rotation=45, ha="right", fontsize=9)
ax.legend(fontsize=9, loc="lower left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top15_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in top_15.iterrows():
    metric = row["metric"]
    clean_idx = df[metric].notna()
    y_score = df.loc[clean_idx, metric].values * row["direction"]
    y_true = df.loc[clean_idx, "binder"].values
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ax.plot(fpr, tpr, linewidth=1.5, label=f"{metric} (AUC={row['aligned_roc']:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Top 15 ROC Curves")
ax.legend(fontsize=6, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(12, 6))
for _, row in top_15.iterrows():
    metric = row["metric"]
    clean_idx = df[metric].notna()
    y_score = df.loc[clean_idx, metric].values * row["direction"]
    y_true = df.loc[clean_idx, "binder"].values
    if len(np.unique(y_true)) < 2:
        continue
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ax.plot(recall, precision, label=f"{metric} (PR-AUC={row['pr_auc']:.3f})")

ax.hlines(df["binder"].mean(), 0, 1, colors="k", linestyles="--", alpha=0.3)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Top 15 Precision-Recall Curves")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
ax.legend(fontsize=6, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 by precision (horizontal)
metric_label_map = {
    "chai_unconstrained_per_chain_iptm": "Chai-1 ipTM",
    "cf_ipsae": "ColabFold ipSAE",
    "cf_pdockq2": "ColabFold pDockQ2",
    "cf_iplddt": "ColabFold ipLDDT",
    "af3_ptm": "AlphaFold3 pTM",
    "cf_interface_dG": "ColabFold Interface dG",
    "af3_ipsae": "AlphaFold3 ipSAE",
    "af3_iptm": "AlphaFold3 ipTM",
    "esmfold_ipae": "ESMFold ipAE",
    "af3_pdockq2": "AlphaFold3 pDockQ2",
}

top_prec = rankings.sort_values("precision", ascending=False).head(10)
y = np.arange(len(top_prec))
labels = [metric_label_map.get(m, m) for m in top_prec["metric"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(y, top_prec["precision"], color="#f39c12", alpha=0.8)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Precision", fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Compare metrics pairwise. reindex guards against score columns being absent
# when some metrics failed to score; dropna drops rows that weren't scored.
score_cols = {
    "ROC-AUC": "aligned_roc",
    "PR-AUC": "pr_auc",
    "F1-Score": "f1",
    "Precision": "precision",
    "Recall": "recall",
}
scored = rankings.reindex(columns=["metric", *score_cols.values()]).dropna(subset=["pr_auc"])
comparison = scored[list(score_cols.values())].rename(columns={v: k for k, v in score_cols.items()})

pairs = [
    ("PR-AUC", "ROC-AUC", "steelblue"),
    ("ROC-AUC", "F1-Score", "seagreen"),
    ("PR-AUC", "F1-Score", "coral"),
    ("Precision", "PR-AUC", "steelblue"),
    ("Precision", "ROC-AUC", "seagreen"),
    ("Precision", "F1-Score", "coral"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (xk, yk, color) in zip(axes.flatten(), pairs):
    ax.scatter(comparison[xk], comparison[yk], alpha=0.6, s=50, color=color)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel(xk); ax.set_ylabel(yk)
    ax.set_title(f"{xk} vs {yk}", fontweight="bold")
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.suptitle("Metric Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

#### Quadrant plots

In [ ]:
# TO DO: constants?
colors_map = {
    'germinal': "#2ecc71",
    'mcmahon': "#d6a439",
    'snir_ab': "#1f3397",
    'harvey': "#e73cd0",
    'peptide': "#2ecc71",
    'alphaseq': "#e72cd0",

}

marker_map = {0: "X", 1: "o"}
marker_map_ply = {0: "x", 1: "circle"}
alpha_map = {"target_shuffle": 0.4, "original": 1.0, "alanine_scan": 1.0}

In [ ]:
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

def plot_top_quadrant(scores_df, ranking_df):
    top_20 = ranking_df.sort_values(by="pr_auc", ascending=False).head(20)
    if len(top_20) < 2:
        return
    m1_info, m2_info = top_20.iloc[0], top_20.iloc[1]
    m1, t1 = m1_info["metric"], m1_info["opt_threshold_raw"]
    m2, t2 = m2_info["metric"], m2_info["opt_threshold_raw"]

    plot_df = (
        scores_df.groupby(["source", "sample", "type", "binder"])[[m1, m2]]
        .agg(["mean", "std"]).reset_index()
    )
    plot_df.columns = [f"{c[0]}_{c[1]}" if c[1] else c[0] for c in plot_df.columns]

    fig, ax = plt.subplots(figsize=(8, 8))
    for t, subdf in plot_df.groupby("type"):
        sns.scatterplot(
            data=subdf, x=f"{m1}_mean", y=f"{m2}_mean",
            hue="source", palette=colors_map, style="binder", markers=marker_map,
            s=80, alpha=alpha_map.get(t, 1.0), edgecolor="w", linewidth=0.6,
            legend=False, ax=ax,
        )
    ax.set_aspect("equal", adjustable="box")
    ax.axvline(t1, color="black", linestyle="--", alpha=0.3)
    ax.axhline(t2, color="black", linestyle="--", alpha=0.3)

    max_val = max(plot_df[f"{m1}_mean"].max(), plot_df[f"{m2}_mean"].max()) * 1.05
    ax.set_xlim(0, max_val); ax.set_ylim(0, max_val)
    ax.set_xlabel(f"{m1}\n(PR-AUC: {m1_info['pr_auc']:.3f})", fontsize=11, labelpad=10)
    ax.set_ylabel(f"{m2}\n(PR-AUC: {m2_info['pr_auc']:.3f})", fontsize=11, labelpad=10)

    handles = [mpatches.Patch(color="none", label="Source")]
    for label, color in colors_map.items():
        handles.append(mlines.Line2D([], [], color=color, marker="s", linestyle="None", markersize=8, label=label))
    ax.legend(handles=handles, fontsize=9, loc="upper left", bbox_to_anchor=(1.05, 0.7), frameon=False)

    plt.tight_layout()
    fig.subplots_adjust(top=0.90, right=0.78)
    plt.show()

In [ ]:
plot_top_quadrant(df, rankings)

In [ ]:
import plotly.graph_objects as go

top_20 = rankings.sort_values(by="pr_auc", ascending=False).head(20)
m1_info, m2_info = top_20.iloc[0], top_20.iloc[1]
m1, t1 = m1_info["metric"], m1_info["opt_threshold_raw"]
m2, t2 = m2_info["metric"], m2_info["opt_threshold_raw"]

plot_df = (
    df.groupby(["source", "sample", "type", "binder"])[[m1, m2]]
    .agg(["mean", "std"]).reset_index()
)
plot_df.columns = [f"{c[0]}_{c[1]}" if c[1] else c[0] for c in plot_df.columns]

fig = go.Figure()
for t, type_df in plot_df.groupby("type"):
    opacity = alpha_map.get(t, 1.0)
    for (source, binder), subdf in type_df.groupby(["source", "binder"]):
        fig.add_trace(go.Scatter(
            x=subdf[f"{m1}_mean"], y=subdf[f"{m2}_mean"], mode="markers",
            marker=dict(color=colors_map[source], symbol=marker_map_ply[binder], size=8, opacity=opacity, line=dict(width=0)),
            name=f"{source} | {t} | binder={binder}", text=subdf["sample"],
            hovertemplate="<b>%{text}</b><br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
        ))
fig.add_vline(x=t1, line_dash="dash", line_color="black", opacity=0.3)
fig.add_hline(y=t2, line_dash="dash", line_color="black", opacity=0.3)
fig.update_layout(width=900, height=700, plot_bgcolor="white",
    title=f"Comparing Metrics: {m1} vs {m2}", xaxis_title=m1, yaxis_title=m2)
fig.show()

## Effect sizes
Scores are direction-aligned (higher = more binder-like). Cliff's delta is the primary non-parametric effect size; AUROC + p-value come from the Mann-Whitney U test.

In [ ]:
from scipy.stats import mannwhitneyu

def compute_auroc_pvalue(df, metrics):
    results = []
    for metric in metrics:
        clean = df[[metric, "binder"]].dropna()
        if clean["binder"].nunique() < 2:
            continue
        y_true = clean["binder"].astype(int)
        score = clean[metric].values * get_direction(metric)   # align: higher = binder-like
        pos, neg = score[y_true == 1], score[y_true == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        U, p = mannwhitneyu(pos, neg, alternative="greater")
        auroc = U / (len(pos) * len(neg))
        results.append({"metric": metric, "auroc": round(auroc, 4), "p_value": round(float(p), 6)})
    return pd.DataFrame(results).sort_values("auroc", ascending=False)

auroc_df = compute_auroc_pvalue(df, numeric_cols)
auroc_df.head(15)

In [ ]:
# Cliff's delta = 2*AUROC - 1  (non-parametric; preferred over Cohen's d here)
def cliffs_delta(df, metrics):
    results = []
    for metric in metrics:
        clean = df[[metric, "binder"]].dropna()
        if clean["binder"].nunique() < 2:
            continue
        y_true = clean["binder"].astype(int)
        score = clean[metric].values * get_direction(metric)
        pos, neg = score[y_true == 1], score[y_true == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        U, _ = mannwhitneyu(pos, neg, alternative="two-sided")
        auroc = U / (len(pos) * len(neg))
        results.append({"metric": metric, "cliffs_delta": round(2 * auroc - 1, 4)})
    return pd.DataFrame(results).sort_values("cliffs_delta", key=abs, ascending=False)

cliff_df = cliffs_delta(df, numeric_cols)
cliff_df.head(15)

## Redundancy
Spearman correlation heatmap (rank-based sanity check on redundancy) followed by PCA / UMAP embeddings of the metrics, colored by family and by model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler


# keep numeric only
X = df.select_dtypes(include=["number"])

# optional: handle inf / NaN
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

In [ ]:
top20 = rankings.head(20)["metric"].tolist()
corr = X[top20].corr(method="spearman")

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="vlag", center=0, square=True, cbar_kws={"shrink": 0.6}, ax=ax)
ax.set_title("Spearman Correlation - Top 20 Metrics")
plt.tight_layout()
plt.show()

In [ ]:
# TO DO: refactor and define constants in constants...
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from umap import UMAP

# --- Data prep ---
X_feat = X.T  # rows = metrics, cols = samples — keep as DataFrame
metric_names = X_feat.index
X_scaled = StandardScaler().fit_transform(X_feat.values)

# --- Strip model prefix --- constants
models_sorted = sorted(
    ['af3', 'cf', 'esmfold', 'chai_constrained', 'chai_unconstrained',
     'boltz_free', 'boltz_template'],
    key=len, reverse=True
)

def strip_model_prefix(metric_name, models_sorted):
    for model in models_sorted:
        if metric_name.startswith(model + "_"):
            return metric_name[len(model) + 1:]
    return metric_name

metric_types = pd.Series(
    [strip_model_prefix(m, models_sorted) for m in metric_names],
    index=metric_names
)

# --- Color map --- constants
metric_family_colors = {
    # Confidence / pLDDT family (blues)
    "plddt":                        "tab:blue",
    "iplddt":                       "steelblue",
    "pred_lddt":                    "cornflowerblue",
    "pred_ilddt":                   "royalblue",
    # PTM / iPTM family (cyans)
    "ptm":                          "tab:cyan",
    "iptm":                         "darkcyan",
    "per_chain_ptm":                "mediumturquoise",
    "per_chain_pair_iptm":          "teal",
    "interface_iptm":               "cadetblue",
    # PAE / error estimates (purples)
    "pae":                          "tab:purple",
    "ipae":                         "mediumpurple",
    "ipsae":                        "orchid",
    "pde":                          "rebeccapurple",
    "ipde":                         "blueviolet",
    # Interface energetics (reds)
    "interface_dG":                 "tab:red",
    "interface_dG_SASA_ratio":      "tomato",
    "interface_energy":             "firebrick",
    "dG":                           "tab:orange",
    # Docking scores (greens)
    "docking_score":                "tab:green",
    "pdockq":                       "limegreen",
    "pdockq2":                      "mediumseagreen",
    "score":                        "darkgreen",
    "confidence_score":             "yellowgreen",
    "aggregate_score":              "olivedrab",
    "pyros_binder_score":           "darkolivegreen",
    "binder_score":                 "tab:olive",
    # Structural / shape (pinks/browns)
    "rmsd":                         "tab:pink",
    "af3_rmsd":                     "hotpink",
    "tm_score":                     "tab:brown",
    "clash_score":                  "sienna",
    "lis":                          "rosybrown",
    # Experimental / biological (yellows)
    "EC50[M]":                      "gold",
    "KD[M]":                        "goldenrod",
    "binder":                       "darkgoldenrod",
    # Fallback
    "other":                        "tab:gray",
}

def assign_color(metric_type, color_map):
    if metric_type in color_map:
        return color_map[metric_type]
    for key in sorted(color_map, key=len, reverse=True):
        if metric_type.startswith(key):
            return color_map[key]
    return color_map["other"]

point_colors = metric_types.map(
    lambda t: assign_color(t, metric_family_colors)
).tolist()

# --- Debug ---
print(f"X_feat shape: {X_feat.shape}")
print(f"n metrics:    {len(metric_types)}")
print("\nStripped metric types:")
print(pd.DataFrame({"original": metric_names, "stripped": metric_types.values}).to_string(index=False))
print("\nType counts:")
print(metric_types.value_counts())

# --- Embeddings ---
pca = PCA(n_components=2, random_state=42)
emb_pca = pca.fit_transform(X_scaled)
print(f"\nPC1: {pca.explained_variance_ratio_[0]:.1%}  PC2: {pca.explained_variance_ratio_[1]:.1%}")

emb_umap = UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(X_scaled)

# --- Legend handles (shared) ---
present_types = metric_types.unique()
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           color=assign_color(t, metric_family_colors), label=t)
    for t in sorted(present_types)
]

# --- Plot helper ---
def plot_embedding(ax, emb, title, xlabel, ylabel, show_labels=True):
    ax.scatter(emb[:, 0], emb[:, 1], c=point_colors, s=60, zorder=2)
    if show_labels:
        for i, name in enumerate(metric_names):
            ax.annotate(name, (emb[i, 0], emb[i, 1]), fontsize=6, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

# --- Figure ---
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

plot_embedding(
    axes[0], emb_pca,
    title="PCA of Metrics — Colored by Metric Type\n(model identity stripped)",
    xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
    ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    # show_labels=False
)

plot_embedding(
    axes[1], emb_umap,
    title="UMAP of Metrics — Colored by Metric Type\n(model identity stripped)",
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    # show_labels=False
)

fig.legend(
    handles=legend_handles,
    title="Metric Type",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    borderaxespad=0,
)
plt.tight_layout()
plt.show()

In [ ]:
# TO DO: refactor and define constants in constants...
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from umap import UMAP

# --- Data prep ---
X_feat = X.T  # rows = metrics, cols = samples — keep as DataFrame
metric_names = X_feat.index
X_scaled = StandardScaler().fit_transform(X_feat.values)

# --- Model assignment (longest prefix first to avoid shadowing) --- constants...
models = [
    'af3', 'cf', 'esmfold', 'esmfold2',
    'chai_constrained', 'chai_unconstrained',
    'boltz_free', 'boltz_template', 'other'
]
colors = {
    "af3":                  "tab:blue",
    "cf":                   "tab:red",
    "esmfold":              "tab:green",
    "esmfold2":             "tab:cyan",
    "chai_constrained":     "tab:purple",
    "chai_unconstrained":   "tab:pink",
    "boltz_free":           "tab:orange",
    "boltz_template":       "tab:brown",
    "other":                "tab:gray",
}
models_sorted = sorted([m for m in models if m != "other"], key=len, reverse=True)

metric_models = []
for metric in metric_names:
    assigned = "other"
    for model in models_sorted:
        if metric.startswith(model + "_") or metric == model:
            assigned = model
            break
    metric_models.append(assigned)

metric_models = pd.Series(metric_models, index=metric_names)
point_colors = metric_models.map(colors).tolist()

# --- Debug ---
print(f"X_feat shape: {X_feat.shape}")
print(f"n metrics:    {len(metric_models)}")
print("\nModel assignments:")
print(pd.DataFrame({"metric": metric_names, "model": metric_models.values}).to_string(index=False))
print("\nModel counts:")
print(metric_models.value_counts())

# --- Embeddings ---
pca = PCA(n_components=2, random_state=42)
emb_pca = pca.fit_transform(X_scaled)
print(f"\nPC1: {pca.explained_variance_ratio_[0]:.1%}  PC2: {pca.explained_variance_ratio_[1]:.1%}")

emb_umap = UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(X_scaled)

# --- Legend handles (shared) ---
present_models = metric_models.unique()
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8, color=colors[m], label=m)
    for m in models  # explicit order, not alphabetical
    if m in present_models
]

# --- Plot helper ---
def plot_embedding(ax, emb, title, xlabel, ylabel, show_labels=False):
    ax.scatter(emb[:, 0], emb[:, 1], c=point_colors, s=60, zorder=2)
    if show_labels:
        for i, name in enumerate(metric_names):
            ax.annotate(name, (emb[i, 0], emb[i, 1]), fontsize=6, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

# --- Figure ---
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

plot_embedding(
    axes[0], emb_pca,
    title="PCA of Metrics — Colored by Model Family",
    xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
    ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    # show_labels=False
)

plot_embedding(
    axes[1], emb_umap,
    title="UMAP of Metrics — Colored by Model Family",
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    # show_labels=Fals
)

fig.legend(
    handles=legend_handles,
    title="Model",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    borderaxespad=0,
)
plt.tight_layout()
plt.show()